In [0]:
from collections import defaultdict
from pyspark.sql.functions import to_date
import re

files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/customer/")
grouped = defaultdict(list)

for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    latest = sorted_files[0]
    old_files = sorted_files[1:]
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        print(f"Table {table_name} does not exist. Creating new table...")
        df_new = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(latest)
        
        if "LastUpdated" in df_new.columns:
            df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
        if "TxnDate" in df_new.columns:
            df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
        
        df_new.write.mode("append").saveAsTable(table_name)
        print(f"Created new table {table_name} with {df_new.count()} records from {latest.split('/')[-1]}")
    else:
        print(f"Table {table_name} exists. Processing new data...")
        df_new = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(latest)
        
        if "LastUpdated" in df_new.columns:
            df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
        if "TxnDate" in df_new.columns:
            df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
        
        try:
            table_type = spark.sql(f"DESCRIBE DETAIL {table_name}").select("format").collect()[0][0]
            is_delta = table_type.lower() == "delta"
        except:
            is_delta = False
        
        if is_delta:
            df_existing = spark.table(table_name)
            if "LastUpdated" in df_existing.columns:
                df_existing = df_existing.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
            if "TxnDate" in df_existing.columns:
                df_existing = df_existing.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
            
            df_unique = df_new.exceptAll(df_existing)
            unique_count = df_unique.count()
            
            if unique_count > 0:
                df_unique.write.mode("append").saveAsTable(table_name)
                print(f"Appended {unique_count} unique records from {latest.split('/')[-1]} to {table_name}")
            else:
                print(f"No new unique records found in {latest.split('/')[-1]}. Skipping append.")
        else:
            print(f"Converting {table_name} from external to Delta table...")
            df_existing = spark.table(table_name)
            if "LastUpdated" in df_existing.columns:
                df_existing = df_existing.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
            if "TxnDate" in df_existing.columns:
                df_existing = df_existing.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
            
            temp_table = f"{table_name}_temp"
            df_existing.write.mode("overwrite").saveAsTable(temp_table)
            existing_count = spark.table(temp_table).count()
            df_unique = df_new.exceptAll(df_existing)
            unique_count = df_unique.count()
            df_combined = spark.table(temp_table).union(df_unique)
            spark.sql(f"DROP TABLE {table_name}")
            df_combined.write.mode("overwrite").saveAsTable(table_name)
            print(f"Converted table and loaded {df_combined.count()} total records (existing: {existing_count}, new unique: {unique_count})")
            spark.sql(f"DROP TABLE {temp_table}")
    
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        dbutils.fs.mv(old_file, f"s3://retail-etl-project-revanth/archive/customer/{file_name}")
    
    print(f"Archived {len(old_files)} old file(s) for {dataset}")

In [0]:
%sql
SELECT count(*)
FROM retail_catalog.bronze.customers_raw

In [0]:
from collections import defaultdict
from pyspark.sql.functions import to_date
import re

files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/products/")
grouped = defaultdict(list)

for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    latest = sorted_files[0]
    old_files = sorted_files[1:]
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    
    df_new = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(latest)
    
    if "LastUpdated" in df_new.columns:
        df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
    if "TxnDate" in df_new.columns:
        df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
    
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        df_new.write.mode("overwrite").saveAsTable(table_name)
        print(f"Created new table {table_name} with {df_new.count()} records from {latest.split('/')[-1]}")
    else:
        temp_view = f"{dataset}_updates"
        df_new.createOrReplaceTempView(temp_view)
        
        merge_sql = f"""
        MERGE INTO {table_name} AS target
        USING {temp_view} AS source
        ON target.ProductID = source.ProductID
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """
        
        result = spark.sql(merge_sql)
        print(f"Merged data from {latest.split('/')[-1]} into {table_name}")
    
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        dbutils.fs.mv(old_file, f"s3://retail-etl-project-revanth/archive/products/{file_name}")
    
    print(f"Archived {len(old_files)} old file(s) for {dataset}")

In [0]:
from collections import defaultdict
from pyspark.sql.functions import to_date
import re

files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/sales/")
grouped = defaultdict(list)

for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    latest = sorted_files[0]
    old_files = sorted_files[1:]
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    
    df_new = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(latest)
    
    if "LastUpdated" in df_new.columns:
        df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
    if "TxnDate" in df_new.columns:
        df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
    
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        df_new.write.mode("overwrite").saveAsTable(table_name)
        print(f"Created new table {table_name} with {df_new.count()} records from {latest.split('/')[-1]}")
    else:
        temp_view = f"{dataset}_updates"
        df_new.createOrReplaceTempView(temp_view)
        
        key_col = None
        for col in df_new.columns:
            if col.lower() in ['salesid', 'transactionid', 'saleid', 'txnid']:
                key_col = col
                break
        
        if key_col:
            merge_sql = f"""
            MERGE INTO {table_name} AS target
            USING {temp_view} AS source
            ON target.{key_col} = source.{key_col}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
            """
            result = spark.sql(merge_sql)
            print(f"Merged data from {latest.split('/')[-1]} into {table_name} using key {key_col}")
        else:
            print(f"No unique key found for {table_name}, appending all records")
            df_new.write.mode("append").saveAsTable(table_name)
    
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        dbutils.fs.mv(old_file, f"s3://retail-etl-project-revanth/archive/sales/{file_name}")
    
    print(f"Archived {len(old_files)} old file(s) for {dataset}")

In [0]:
from collections import defaultdict
from pyspark.sql.functions import to_date
import re

files = dbutils.fs.ls("s3://retail-etl-project-revanth/incoming/stores/")
grouped = defaultdict(list)

for f in files:
    dataset = f.name.split("_")[0]
    grouped[dataset].append(f.path)

def extract_ts(file):
    return re.search(r'_(\d{8}_\d{6})', file).group(1)

for dataset, file_list in grouped.items():
    sorted_files = sorted(file_list, key=extract_ts, reverse=True)
    latest = sorted_files[0]
    old_files = sorted_files[1:]
    table_name = f"retail_catalog.bronze.{dataset}_raw"
    
    df_new = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(latest)
    
    if "LastUpdated" in df_new.columns:
        df_new = df_new.withColumn("LastUpdated", to_date("LastUpdated", "d-M-yyyy"))
    if "TxnDate" in df_new.columns:
        df_new = df_new.withColumn("TxnDate", to_date("TxnDate", "d-M-yyyy"))
    
    table_exists = spark.catalog.tableExists(table_name)
    
    if not table_exists:
        df_new.write.mode("overwrite").saveAsTable(table_name)
        print(f"Created new table {table_name} with {df_new.count()} records from {latest.split('/')[-1]}")
    else:
        temp_view = f"{dataset}_updates"
        df_new.createOrReplaceTempView(temp_view)
        
        merge_sql = f"""
        MERGE INTO {table_name} AS target
        USING {temp_view} AS source
        ON target.StoreID = source.StoreID
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """
        
        result = spark.sql(merge_sql)
        print(f"Merged data from {latest.split('/')[-1]} into {table_name}")
    
    for old_file in old_files:
        file_name = old_file.split("/")[-1]
        dbutils.fs.mv(old_file, f"s3://retail-etl-project-revanth/archive/stores/{file_name}")
    
    print(f"Archived {len(old_files)} old file(s) for {dataset}")